#### v1 jewelry_trends_scraper.py 22dec

#### v2 jewelry_trends_scraper_duckduckgo.py 22dec

 #### v3 agent.py  23dec

#### v4 agent_oxylabs.py 23dec

#### v5 agent_scrapegraph.py 23dec

#### v6 Duck Duck go, webcrawl without agno 24,25dec

In [ ]:
#!/usr/bin/env python3

from ddgs import DDGS
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin, urlparse
import time

# -----------------------------
# CONFIG
# -----------------------------
USER_AGENT = "Mozilla/5.0 (Windows NT 10.0; Win64; x64)"
HEADERS = {"User-Agent": USER_AGENT}
IMAGE_EXTENSIONS = (".jpg", ".jpeg", ".png", ".webp")
MAX_SITES = 5
DELAY_BETWEEN_REQUESTS = 1.5  # seconds


# -----------------------------
# STEP 1: DuckDuckGo → WEBSITE URLs
# -----------------------------
def ddg_website_search(query: str, max_results: int = 5):
    urls = []
    with DDGS() as ddgs:
        results = ddgs.text(query, max_results=max_results)

        for r in results:
            if r.get("href"):
                urls.append(r["href"])

    return urls


# -----------------------------
# STEP 2: Visit website → extract image URLs
# -----------------------------
def extract_images_from_website(url: str):
    images = set()

    try:
        response = requests.get(url, headers=HEADERS, timeout=10)
        response.raise_for_status()
    except Exception as e:
        print(f"[ERROR] Failed to fetch {url}: {e}")
        return []

    soup = BeautifulSoup(response.text, "html.parser")

    # Find all <img> tags
    for img in soup.find_all("img"):
        src = img.get("src")
        if not src:
            continue

        full_url = urljoin(url, src)

        if full_url.lower().endswith(IMAGE_EXTENSIONS):
            images.add(full_url)

    return list(images)


# -----------------------------
# STEP 3: Main pipeline
# -----------------------------
def get_images_from_query(query: str):
    print(f"\n🔍 Searching websites for: {query}\n")

    websites = ddg_website_search(query, MAX_SITES)
    all_images = set()

    for site in websites:
        print(f"🌐 Crawling: {site}")
        images = extract_images_from_website(site)

        print(f"   ↳ Found {len(images)} images")
        for img in images:
            all_images.add(img)

        time.sleep(DELAY_BETWEEN_REQUESTS)

    return list(all_images)


# -----------------------------
# RUN
# -----------------------------
if __name__ == "__main__":
    query = "latest gold pendant jewelry designs"

    image_urls = get_images_from_query(query)

    print("\n✅ FINAL IMAGE URLs\n")
    for url in image_urls:
        print(url)

#### v7 agent agno_v6 26dec
Agno Agent (LLM)
→ decides refined search query + filters
→ calls deterministic tools
→ tools fetch websites
→ tools extract images
→ agent returns final image URLs

pip install agno duckduckgo-search requests beautifulsoup4


In [ ]:
from ddgs import DDGS
SOURCE= "giva.co"
def ddg_website_search(query: str, max_results: int = 5):
    """
    Search DuckDuckGo for website URLs.
    """
    urls = []
    with DDGS() as ddgs:
        results = ddgs.text(query, max_results=max_results)

        for r in results:
            if r.get("href"):
                urls.append(r["href"])

    return urls

import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin

IMAGE_EXTENSIONS = (".jpg", ".jpeg", ".png", ".webp")
HEADERS = {"User-Agent": "Mozilla/5.0"}


def extract_images_from_website(url: str):
    """
    Crawl a website and extract direct image URLs.
    """
    images = set()

    try:
        response = requests.get(url, headers=HEADERS, timeout=10)
        response.raise_for_status()
    except Exception:
        return []

    soup = BeautifulSoup(response.text, "html.parser")

    for img in soup.find_all("img"):
        src = img.get("src")
        if not src:
            continue

        full_url = urljoin(url, src)
        if full_url.lower().endswith(IMAGE_EXTENSIONS):
            images.add(full_url)

    return list(images)

from agno.agent import Agent
from agno.models.huggingface import HuggingFace
import os

HF_TOKEN = os.getenv("HF_TOKEN")

agent = Agent(
    name="Jewelry Image Intelligence Agent",
    model=HuggingFace(
        id="HuggingFaceTB/SmolLM3-3B",
        api_key=HF_TOKEN,
    ),
    tools=[
        ddg_website_search,
        extract_images_from_website
    ],
    instructions=[
        "Your job is to find jewelry pendant images.",
        "Step 1: Call ddg_website_search with a refined query.",
        "Step 2: For each returned website URL, call extract_images_from_website.",
        "Do not explain anything.",
        "Return only final image URLs.",
        "Do not hallucinate URLs."
    ],
    markdown=False
)

def agentic_image_pipeline(user_query: str):
    # Step 1: Agent decides search query
    refined_query_prompt = f"""
    Refine this query for discovering jewelry pendant websites:
    "{user_query}"
    Return only the refined query text.
    """

    refined_query = agent.run(refined_query_prompt).content
    if not refined_query:
        refined_query = user_query

    # Step 2: Get websites
    websites = ddg_website_search(refined_query, max_results=5)

    # Step 3: Crawl websites
    all_images = set()
    for site in websites:
        images = extract_images_from_website(site)
        for img in images:
            all_images.add(img)

    return list(all_images)


In [ ]:
if __name__ == "__main__":
    query = "latest gold butterfly pendant jewelry designs"

    image_urls = agentic_image_pipeline(query)

    print("\n✅ FINAL IMAGE URLs\n")
    for url in image_urls:
        print(url)



<span style="color: pink">The uncomfortable truth (said plainly)  
❌ Agentic ≠ Omniscient  
❌ An LLM cannot see images  
❌ It cannot judge relevance from a URL alone  
So when you crawl a website and extract <img> tags, you will always get:  
banners,logos,icons,tracking pixels,social buttons,unrelated images,.This is not an intelligence failure.It’s a missing perception layer.</span>

#### v8 Filtered img crawling on v7 27dec

Raw images
↓
Structural filters (fast, deterministic)
↓
Semantic filters (agentic reasoning)
↓
Visual filters (optional, best)

In [2]:
#!/usr/bin/env python3

import os
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin
from ddgs import DDGS

from agno.agent import Agent
from agno.models.huggingface import HuggingFace

# ================= CONFIG =================
HF_TOKEN = os.getenv("HF_TOKEN")

HEADERS = {
    "User-Agent": "Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 Chrome/120"
}

IMAGE_EXTS = (".jpg", ".jpeg", ".png", ".webp")
BAD_KEYWORDS = [
    "logo", "icon", "sprite", "favicon", "avatar",
    "banner", "header", "footer", "ads"
]

# ================= AGENT =================
agent = Agent(
    name="Jewelry Pendant Image Agent",
    model=HuggingFace(
        id="HuggingFaceTB/SmolLM3-3B",
        api_key=HF_TOKEN
    ),
    instructions=[
        "You are a jewelry product expert.",
        "Select ONLY clear product or design images of jewelry pendants.",
        "Reject logos, UI elements, banners, people, lifestyle shots.",
        "Return ONLY valid image URLs.",
        "One URL per line. No explanation."
    ],
    markdown=False
)

# ================= HTTP SESSION =================
session = requests.Session()
session.headers.update(HEADERS)

# ================= SEARCH =================
def duckduckgo_website_search(query: str, max_results: int = 5):
    urls = []
    with DDGS() as ddgs:
        results = ddgs.text(query, max_results=max_results)
        for r in results:
            if r.get("href"):
                urls.append(r["href"])
    return urls

# ================= IMAGE EXTRACTION =================
def extract_images_with_context(url: str):
    images = []

    try:
        with session.get(url, timeout=10, stream=True) as response:
            response.raise_for_status()
            html = response.text
    except Exception:
        return images

    soup = BeautifulSoup(html, "html.parser")

    for img in soup.find_all("img"):
        src = img.get("src")
        if not src:
            continue

        full_url = urljoin(url, src)

        if not full_url.lower().endswith(IMAGE_EXTS):
            continue

        if any(bad in full_url.lower() for bad in BAD_KEYWORDS):
            continue

        alt = img.get("alt", "").strip()
        context = img.parent.get_text(" ", strip=True)[:300]

        images.append({
            "url": full_url,
            "alt": alt,
            "context": context
        })

    return images

# ================= AGENTIC FILTER =================
def agent_filter_images(images):
    if not images:
        return []

    prompt = f"""
From the following image data, select ONLY images that are jewelry pendants.

Rules:
- Must be a jewelry pendant product or design
- No logos, no UI, no banners, no people
- Ignore irrelevant objects

Return ONLY the image URLs.

Images:
{images}
"""

    response = agent.run(prompt)

    if response.content:
        return [
            line.strip()
            for line in response.content.splitlines()
            if line.strip().startswith("http")
        ]

    return []

# ================= PIPELINE =================
def get_relevant_pendant_images(query: str):
    websites = duckduckgo_website_search(query, max_results=5)

    collected_images = []
    for site in websites:
        collected_images.extend(extract_images_with_context(site))

    return agent_filter_images(collected_images)

# ================= RUN =================
if __name__ == "__main__":
    query = "latest gold butterfly pendant jewelry designs"

    results = get_relevant_pendant_images(query)

    print("\n✅ FINAL RELEVANT PENDANT IMAGE URLs\n")
    for url in results:
        print(url)


ERROR    HF_TOKEN not set. Please set the HF_TOKEN environment variable.

ERROR    Unexpected error invoking HuggingFace model: You must provide an api_key to work with auto API or log in  
         with `hf auth login`.

ERROR    Error in Agent run: You must provide an api_key to work with auto API or log in with `hf auth login`.


✅ FINAL RELEVANT PENDANT IMAGE URLs



#### v9 
27dec

In [ ]:
import os
from dotenv import load_dotenv
load_dotenv()
from agno.agent import Agent
from agno.models.ollama import Ollama
from ddgs import DDGS
from textwrap import dedent

# 1. Updated Scraper to handle multiple brands
def get_brand_jewelry_images(query: str, count: int =10) -> str:
    """
    Fetches jewelry image URLs from specific brands like Kalyan, Giva, and Palmonas.
    """
    results_list = []
    # List of targeted brands
    brands = ["Kalyan Jewelers", "GIVA Silver", "Palmonas"]

    try:
        with DDGS() as ddgs:
            # We construct a query that targets these specific sites
            # Example: "latest silver bangles (site:kalyanjewellers.net OR site:giva.co OR site:palmonas.com)"
            site_filter = "(site:kalyanjewellers.net OR site:giva.co OR site:palmonas.com OR site:pinterest.com)"
            # site_filter = "(site:giva.co OR site:palmonas.com OR site:pinterest.com)"

            combined_query = f"{query} {site_filter}"

            image_search = ddgs.images(
                query = combined_query,
                keywords=combined_query,
                region="wt-wt",
                safesearch="moderate",
                size="Medium",
                timelimit="d",
                type_image="photo",
                max_results=count
            )

            for r in image_search:
                # We return only the image link as per your request
                results_list.append(r['image'])

        return str(results_list) if results_list else "[]"
    except Exception as e:
        return f"Error: {str(e)}"

# 2. Setup the Agent
agent = Agent(
    model=Ollama(id="qwen2.5:7b"),
    tools=[get_brand_jewelry_images],
    instructions=dedent("""
       - You are a precise data assistant.
        - When the user asks for images, call 'get_jewelry_image_links'.
        - ONLY provide the raw URLs of the images.
        - Do not add descriptions or markdown formatting unless asked.
        - Please only output in json format.
    """),
    markdown=True
)

# 3. Execution
query = "gold ring"
agent.print_response(query)
res = agent.run(query).content
import json
res = json.loads(res)
print(res)

from IPython.display import Image, display

# Assuming 'res' is your list: ['url1', 'url2', ...]
print(f"Displaying {len(res)} images from the list:")

for link in res:
    try:
        # We use url=link to fetch the remote image
        # You can adjust width (e.g., 300) to make them smaller
        display(Image(url=link, width=400))
    except Exception as e:

        print(f"Could not load image from {link}: {e}")

/home/ec2-user/fusion_engine/.venv/lib64/python3.12/site-packages/rich/live.py:256: UserWarning: install 
"ipywidgets" for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

JSONDecodeError: Expecting value: line 1 column 1 (char 0)

#### v10,selenium+Agno

In [6]:
#!/usr/bin/env python3

import os
import time
from dotenv import load_dotenv
load_dotenv()
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from bs4 import BeautifulSoup
from agno.agent import Agent
from agno.models.huggingface import HuggingFace

# ================= CONFIG =================
HF_TOKEN = os.getenv("HF_TOKEN")

IMAGE_EXTS = (".jpg", ".jpeg", ".png", ".webp")
BAD_KEYWORDS = ["logo", "icon", "sprite", "favicon", "banner", "ads", "avatar"]

# ================= AGENT =================
agent = Agent(
    name="Jewelry Pendant Image Agent",
    model=HuggingFace(
        id="HuggingFaceTB/SmolLM3-3B",
        api_key=HF_TOKEN
    ),
    instructions=[
        "You are a jewelry product expert.",
        "Select ONLY clear product or design images of jewelry pendants.",
        "Reject logos, UI elements, banners, people, lifestyle shots.",
        "Return ONLY valid image URLs.",
        "One URL per line. No explanation."
    ],
    markdown=False
)

# ================= SELENIUM SETUP =================
def init_driver():
    options = Options()
    options.add_argument("--headless")
    options.add_argument("--disable-gpu")
    options.add_argument("--no-sandbox")
    options.add_argument("--disable-dev-shm-usage")
    driver = webdriver.Chrome(options=options)
    return driver

# ================= IMAGE EXTRACTION =================
def extract_images_from_page(driver, url, limit=20):
    driver.get(url)
    time.sleep(3)  # allow JS to load
    soup = BeautifulSoup(driver.page_source, "html.parser")

    images = []
    for img in soup.find_all("img"):
        src = img.get("src")
        if not src:
            continue
        if not src.lower().endswith(IMAGE_EXTS):
            continue
        if any(bad in src.lower() for bad in BAD_KEYWORDS):
            continue
        images.append(src)
        if len(images) >= limit:  # stop early
            break
    return images

# ================= AGENT FILTER =================
def agent_filter_images(images, max_results=5):
    if not images:
        return []
    prompt = f"""
From the following image data, select ONLY images that are jewelry pendants.

Rules:
- Must be a jewelry pendant product or design
- No logos, no UI, no banners, no people
- Ignore irrelevant objects

Return ONLY the image URLs.

Images:
{images}
"""
    response = agent.run(prompt)
    if response.content:
        urls = [
            line.strip()
            for line in response.content.splitlines()
            if line.strip().startswith("http")
        ]
        return urls[:max_results]  # enforce max 5
    return []

# ================= PIPELINE =================
def crawl_site(driver, site_name, collection_url):
    print(f"\n🔎 Crawling {site_name}...")
    raw_images = extract_images_from_page(driver, collection_url, limit=20)
    filtered = agent_filter_images(raw_images, max_results=5)
    return filtered

# ================= RUN =================
if __name__ == "__main__":
    driver = init_driver()

    sites = {
        "Giva": "https://www.giva.co/collections/pendants",
        "Kalyan": "https://www.kalyanjewellers.net/jewellery/pendants.php",
        "Malabar Golds": "https://www.malabargoldanddiamonds.com/jewellery/pendants.html"
    }

    all_results = {}
    for name, url in sites.items():
        all_results[name] = crawl_site(driver, name, url)

    driver.quit()

    print("\n✅ FINAL PENDANT IMAGE URLs (5 per site)\n")
    for site, urls in all_results.items():
        print(f"\n--- {site} ---")
        for u in urls:
            print(u)



🔎 Crawling Giva...

🔎 Crawling Kalyan...

🔎 Crawling Malabar Golds...

✅ FINAL PENDANT IMAGE URLs (5 per site)


--- Giva ---

--- Kalyan ---

--- Malabar Golds ---
https://static.malabargoldanddiamonds.com/media/catalog/category/hoops.jpg


#### v11 5 images+selenium+agno+no static image 29dec

In [8]:
#!/usr/bin/env python3

import os
from dotenv import load_dotenv
load_dotenv()
import time
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from bs4 import BeautifulSoup
from agno.agent import Agent
from agno.models.huggingface import HuggingFace

# ================= CONFIG =================
HF_TOKEN = os.getenv("HF_TOKEN")

IMAGE_EXTS = (".jpg", ".jpeg", ".png", ".webp")
BAD_KEYWORDS = ["logo", "icon", "sprite", "favicon", "banner", "ads", "avatar"]

# ================= AGENT =================
agent = Agent(
    name="Jewelry Pendant Image Agent",
    model=HuggingFace(
        id="HuggingFaceTB/SmolLM3-3B",
        api_key=HF_TOKEN
    ),
    instructions=[
        "You are a jewelry product expert.",
        "Select ONLY clear product or design images of jewelry pendants.",
        "Reject logos, UI elements, banners, people, lifestyle shots.",
        "Return ONLY valid image URLs.",
        "One URL per line. No explanation."
    ],
    markdown=False
)

# ================= SELENIUM SETUP =================
def init_driver():
    options = Options()
    options.add_argument("--headless")
    options.add_argument("--disable-gpu")
    options.add_argument("--no-sandbox")
    options.add_argument("--disable-dev-shm-usage")
    driver = webdriver.Chrome(options=options)
    return driver

# ================= IMAGE EXTRACTION =================
def extract_images(driver, url, selector, limit=20):
    driver.get(url)
    time.sleep(7)
    driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
    time.sleep(7)

    soup = BeautifulSoup(driver.page_source, "html.parser")
    images = []

    for img in soup.select(selector):
        src = img.get("src")
        if not src:
            continue
        if not src.lower().endswith(IMAGE_EXTS):
            continue
        if any(bad in src.lower() for bad in BAD_KEYWORDS):
            continue
        images.append(src)
        if len(images) >= limit:
            break

    return images

# ================= AGENT FILTER =================
def agent_filter_images(images, max_results=5):
    if not images:
        return []
    prompt = f"""
From the following image data, select ONLY images that are jewelry pendants.

Rules:
- Must be a jewelry pendant product or design
- No logos, no UI, no banners, no people
- Ignore irrelevant objects

Return ONLY the image URLs.

Images:
{images}
"""
    response = agent.run(prompt)
    if response.content:
        urls = [
            line.strip()
            for line in response.content.splitlines()
            if line.strip().startswith("http")
        ]
        return urls[:max_results]
    return []

# ================= PIPELINE =================
def crawl_site(driver, site_name, collection_url, selector):
    print(f"\n🔎 Crawling {site_name}...")
    raw_images = extract_images(driver, collection_url, selector, limit=20)
    filtered = agent_filter_images(raw_images, max_results=5)
    return filtered

# ================= RUN =================
if __name__ == "__main__":
    driver = init_driver()

    sites = {
        "Giva": {
            "url": "https://www.giva.co/collections/pendants",
            "selector": "img[src*='cdn/shop/files']"
        },
        "Kalyan": {
            "url": "https://www.kalyanjewellers.net/jewellery/pendants.php",
            "selector": "img.product-image"
        },
        "Malabar Golds": {
            "url": "https://www.malabargoldanddiamonds.com/jewellery/pendants.html",
            "selector": "img.product-image-photo"
        }
    }

    all_results = {}
    for name, profile in sites.items():
        all_results[name] = crawl_site(driver, name, profile["url"], profile["selector"])

    driver.quit()

    print("\n✅ FINAL PENDANT IMAGE URLs (5 per site)\n")
    for site, urls in all_results.items():
        print(f"\n--- {site} ---")
        for u in urls:
            print(u)



🔎 Crawling Giva...

🔎 Crawling Kalyan...

🔎 Crawling Malabar Golds...

✅ FINAL PENDANT IMAGE URLs (5 per site)


--- Giva ---

--- Kalyan ---

--- Malabar Golds ---


trying out bluestone

In [1]:
! pip install agno ddgs selenium webdriver_manager dotenv

  Using cached hyperframe-6.1.0-py3-none-any.whl.metadata (4.3 kB)
  Using cached hpack-4.1.0-py3-none-any.whl.metadata (4.6 kB)
   ---------------------------------------- 0.0/1.6 MB ? eta -:--:--
   -------- ------------------------------- 0.4/1.6 MB 10.9 MB/s eta 0:00:01
   --------------------------- ------------ 1.1/1.6 MB 17.7 MB/s eta 0:00:01
   ---------------------------------------  1.6/1.6 MB 17.0 MB/s eta 0:00:01
   ---------------------------------------- 1.6/1.6 MB 14.5 MB/s eta 0:00:00
   ---------------------------------------- 0.0/40.3 kB ? eta -:--:--
   ---------------------------------------- 40.3/40.3 kB ? eta 0:00:00
   ---------------------------------------- 0.0/161.7 kB ? eta -:--:--
   ---------------------------------------- 161.7/161.7 kB 9.5 MB/s eta 0:00:00
   ---------------------------------------- 0.0/3.1 MB ? eta -:--:--
   ------------- -------------------------- 1.1/3.1 MB 33.0 MB/s eta 0:00:01
   ---------------------------- ----------- 2.3/3.1 MB 2


[notice] A new release of pip is available: 24.1 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [1]:
import csv
from selenium import webdriver
import time
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import TimeoutException

# Create an instance of Options
chrome_options = Options()
chrome_options.add_experimental_option("detach", True)
chrome_options.add_argument("--disable-notifications")

# Initialize WebDriver with ChromeDriverManager and the options
driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=chrome_options)
driver.maximize_window()
driver.get("https://www.bluestone.com/")
print("This is the data of bluestone......")
driver.implicitly_wait(3)

# Function to perform search
def search(keyword):
    search_input = driver.find_element(By.XPATH, "/html/body/header/div/div[1]/div[2]/div/div/div[2]/div/div/div/div/form/input[6]")
    search_input.send_keys(keyword, Keys.ENTER)

# Function to retrieve total item count
def Total_item():
    try:
        item_no_element = WebDriverWait(driver, 10).until(EC.visibility_of_element_located((By.XPATH, "/html/body/div[1]/div[2]/div[3]/div/span")))
        text = item_no_element.text.strip()
        # Extracting only the digits from the text
        total_item_no = ''.join(filter(str.isdigit, text))
    except TimeoutException:
        print("Timeout: Element not found or not visible")
        total_item_no = "N/A"
    return total_item_no



# Function to navigate back to main page
def Time_Back():
    time.sleep(5)
    driver.get("https://www.bluestone.com/")
    time.sleep(3)

# Create a list to store the data
data = []

# Perform searches and retrieve total item counts
keywords = ["Men", "Women" , "Rings", "Men's Rings", "Women's Rings", "Earring"]

for keyword in keywords:
    search(keyword)
    item_count = Total_item()
    data.append([keyword, item_count])
    Time_Back()

# Write data to CSV file in a specific directory
csv_file_path = 'E:\\Funndynamix\\04_Project Fusion\\jewelry_data.csv'
with open(csv_file_path, 'w', newline='') as file:
    writer = csv.writer(file)
    writer.writerow(['Keyword', 'Total Item Count'])
    writer.writerows(data)

print("Data saved to", csv_file_path)
driver.quit()  # Quit the WebDriver session

This is the data of bluestone......
Data saved to E:\Funndynamix\04_Project Fusion\my-ec2\jewelry_data.csv


#### v12 images are output-ed , result: most of them are broken image/not available to scrape

In [ ]:
import os
import requests
from selenium import webdriver
import time
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.common.exceptions import StaleElementReferenceException
from webdriver_manager.chrome import ChromeDriverManager

# Define a function to save images
def save_image(src, index):
    img_dir = r'E:\Funndynamix\04_Project Fusion\images'
    if not os.path.exists(img_dir):
        os.makedirs(img_dir)
    response = requests.get(src)
    if response.status_code == 200:
        file_path = os.path.join(img_dir, f"image_{index}.jpg")
        with open(file_path, 'wb') as file:
            file.write(response.content)
            print(f"Image {index} saved successfully at: {file_path}")
    else:
        print(f"Failed to download image {index} from {src}, status code: {response.status_code}")


# Create an instance of Options
chrome_options = Options()
chrome_options.add_experimental_option("detach", True)
chrome_options.add_argument("--disable-notifications")

# Initialize WebDriver with ChromeDriverManager and the options
driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=chrome_options)
driver.maximize_window()
driver.get("https://www.bluestone.com/search")
print("This is the image data of bluestone ......")
driver.implicitly_wait(10)

# Define a function to perform the search
def search(keyword):
    count=0
    while True:
        count+=1
        search_input = driver.find_element(By.XPATH, "/html/body/header/div/div[1]/div[2]/div/div/div[2]/div/div/div/div/form/input[6]")
        search_input.send_keys(keyword, Keys.ENTER)
        if count==5:
            break
        
def total_img_item(max_scrolls=3):
    last_height = driver.execute_script("return document.body.scrollHeight")
    for _ in range(max_scrolls):
        driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
        time.sleep(2)
        new_height = driver.execute_script("return document.body.scrollHeight")
        if new_height == last_height:
            break
        last_height = new_height

    # Collect product images
    img_elements = driver.find_elements(By.CSS_SELECTOR, "ul.product-grid img")
    for index, img in enumerate(img_elements):
        try:
            src = img.get_attribute("src")
            if src and src.endswith((".jpg", ".jpeg", ".png")):
                save_image(src, index)
        except StaleElementReferenceException:
            continue


# Perform searches and retrieve images
keywords = ["Rings","Earring","Neckalce"]

for keyword in keywords:
    search(keyword)
    total_img_item()

# Close the WebDriver session
driver.quit()


#pid_37454 > img

This is the image data of bluestone ......
Image 0 saved successfully at: E:\Funndynamix\04_Project Fusion\images\image_0.jpg
Image 1 saved successfully at: E:\Funndynamix\04_Project Fusion\images\image_1.jpg
Image 2 saved successfully at: E:\Funndynamix\04_Project Fusion\images\image_2.jpg
Image 3 saved successfully at: E:\Funndynamix\04_Project Fusion\images\image_3.jpg
Image 4 saved successfully at: E:\Funndynamix\04_Project Fusion\images\image_4.jpg
Image 5 saved successfully at: E:\Funndynamix\04_Project Fusion\images\image_5.jpg
Image 6 saved successfully at: E:\Funndynamix\04_Project Fusion\images\image_6.jpg
Image 7 saved successfully at: E:\Funndynamix\04_Project Fusion\images\image_7.jpg
Image 8 saved successfully at: E:\Funndynamix\04_Project Fusion\images\image_8.jpg
Image 9 saved successfully at: E:\Funndynamix\04_Project Fusion\images\image_9.jpg
Image 10 saved successfully at: E:\Funndynamix\04_Project Fusion\images\image_10.jpg
Image 11 saved successfully at: E:\Funndyn

KeyboardInterrupt: 

#### v13 fixinf image retrieval
Instead of only src, also look for data-src, data-original, or data-srcset\
Restrict your search to product grid containers\
Use requests.head() to confirm the image is real\

In [5]:
import os
from dotenv import load_dotenv
load_dotenv()
import requests
import time
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.common.exceptions import StaleElementReferenceException
from webdriver_manager.chrome import ChromeDriverManager

# ========== HELPERS ==========
def is_valid_image(url):
    """Check if URL points to a real image."""
    try:
        r = requests.head(url, timeout=5)
        return r.status_code == 200 and "image" in r.headers.get("Content-Type", "")
    except Exception:
        return False

def save_image(src, index, keyword):
    """Download and save image locally."""
    base_dir = r"E:\Funndynamix\04_Project Fusion\images" 
    keyword_dir = os.path.join(base_dir, keyword) 
    os.makedirs(keyword_dir, exist_ok=True)

    response = requests.get(src, timeout=10)
    if response.status_code == 200:
        file_path = os.path.join(keyword_dir, f"image_{index}.jpg")
        with open(file_path, "wb") as f:
            f.write(response.content)
        print(f"✅ Image {index} saved: {file_path}")
    else:
        print(f"❌ Failed to download {src}")

# ========== SELENIUM SETUP ==========
chrome_options = Options()
chrome_options.add_argument("--disable-notifications")
driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=chrome_options)
driver.maximize_window()
driver.get("https://www.bluestone.com/search")
driver.implicitly_wait(5)

# ========== SEARCH FUNCTION ==========
def search(keyword):
    count=0
    while True:
        count+=1
        search_input = driver.find_element(By.XPATH, "/html/body/header/div/div[1]/div[2]/div/div/div[2]/div/div/div/div/form/input[6]")
        search_input.send_keys(keyword, Keys.ENTER)
        if count==5:
            break

# ========== IMAGE EXTRACTION ==========
def total_img_item(keyword,max_scrolls=3, max_images=20):
    last_height = driver.execute_script("return document.body.scrollHeight")
    for _ in range(max_scrolls):
        driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
        time.sleep(2)
        new_height = driver.execute_script("return document.body.scrollHeight")
        if new_height == last_height:
            break
        last_height = new_height

    img_elements = driver.find_elements(By.CSS_SELECTOR, "ul.product-grid li img")
    count = 0
    for index, img in enumerate(img_elements):
        try:
            src = (
                img.get_attribute("src")
                or img.get_attribute("data-src")
                or img.get_attribute("data-original")
            )
            if src and src.endswith((".jpg", ".jpeg", ".png")) and is_valid_image(src):
                save_image(src, index, keyword)
                count += 1
                if count >= max_images:
                    break
        except StaleElementReferenceException:
            continue

# ========== RUN ==========
keywords = ["Rings", "Earring", "Necklace"]

for keyword in keywords:
    print(f"\n🔎 Searching for {keyword}...")
    search(keyword)
    total_img_item(keyword,max_scrolls=3, max_images=5)  # limit to 5 per keyword

driver.quit()



🔎 Searching for Rings...
✅ Image 0 saved: E:\Funndynamix\04_Project Fusion\images\Rings\image_0.jpg
✅ Image 1 saved: E:\Funndynamix\04_Project Fusion\images\Rings\image_1.jpg
✅ Image 2 saved: E:\Funndynamix\04_Project Fusion\images\Rings\image_2.jpg
✅ Image 3 saved: E:\Funndynamix\04_Project Fusion\images\Rings\image_3.jpg
✅ Image 4 saved: E:\Funndynamix\04_Project Fusion\images\Rings\image_4.jpg

🔎 Searching for Earring...
✅ Image 0 saved: E:\Funndynamix\04_Project Fusion\images\Earring\image_0.jpg
✅ Image 1 saved: E:\Funndynamix\04_Project Fusion\images\Earring\image_1.jpg
✅ Image 2 saved: E:\Funndynamix\04_Project Fusion\images\Earring\image_2.jpg
✅ Image 3 saved: E:\Funndynamix\04_Project Fusion\images\Earring\image_3.jpg
✅ Image 4 saved: E:\Funndynamix\04_Project Fusion\images\Earring\image_4.jpg

🔎 Searching for Necklace...
✅ Image 0 saved: E:\Funndynamix\04_Project Fusion\images\Necklace\image_0.jpg
✅ Image 1 saved: E:\Funndynamix\04_Project Fusion\images\Necklace\image_1.jpg
✅

have to make the images which are not avaialbe, delete it or not ssaave it, also only image urls

#### v14 download image urls ina csv

Save the image URLs into a CSV file (with keyword + URL).\
Download the actual image into a folder named after the keyword (so you have the file locally).\
Record the saved filename in the CSV so you know which file corresponds to which URL.\

In [7]:
import os
import csv
import time
import requests
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.common.exceptions import StaleElementReferenceException
from webdriver_manager.chrome import ChromeDriverManager

# ========== CONFIG ==========
BASE_DIR = r"E:\Funndynamix\04_Project Fusion\v14Images"
os.makedirs(BASE_DIR, exist_ok=True)
CSV_PATH = os.path.join(BASE_DIR, "bluestone_urls.csv")

# ========== HELPERS ==========
def is_valid_image(url):
    try:
        r = requests.head(url, timeout=5)
        return r.status_code == 200 and "image" in r.headers.get("Content-Type", "")
    except Exception:
        return False

def save_image(src, index, keyword):
    """Download image into keyword folder and return filename."""
    keyword_dir = os.path.join(BASE_DIR, keyword)
    os.makedirs(keyword_dir, exist_ok=True)

    filename = f"{keyword}_{index}.jpg"
    file_path = os.path.join(keyword_dir, filename)

    response = requests.get(src, timeout=10)
    if response.status_code == 200:
        with open(file_path, "wb") as f:
            f.write(response.content)
        print(f"✅ Saved {filename} for {keyword}")
        return filename
    else:
        print(f"❌ Failed to download {src}")
        return None

# ========== SELENIUM SETUP ==========
chrome_options = Options()
chrome_options.add_argument("--disable-notifications")
driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=chrome_options)
driver.maximize_window()
driver.get("https://www.bluestone.com/search")
driver.implicitly_wait(5)

def search(keyword):
    count=0
    while True:
        count+=1
        search_input = driver.find_element(By.XPATH, "/html/body/header/div/div[1]/div[2]/div/div/div[2]/div/div/div/div/form/input[6]")
        search_input.send_keys(keyword, Keys.ENTER)
        if count==5:
            break

def collect_images(keyword, max_scrolls=3, max_images=5):
    last_height = driver.execute_script("return document.body.scrollHeight")
    for _ in range(max_scrolls):
        driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
        time.sleep(2)
        new_height = driver.execute_script("return document.body.scrollHeight")
        if new_height == last_height:
            break
        last_height = new_height

    img_elements = driver.find_elements(By.CSS_SELECTOR, "ul.product-grid li img")
    results = []
    count = 0
    for index, img in enumerate(img_elements):
        try:
            src = (
                img.get_attribute("src")
                or img.get_attribute("data-src")
                or img.get_attribute("data-original")
            )
            if src and src.endswith((".jpg",".png",".jpeg")) and is_valid_image(src):
                filename = save_image(src, index, keyword)
                if filename:
                    results.append([keyword, src, filename])
                count += 1
                if count >= max_images:
                    break
        except StaleElementReferenceException:
            continue
    return results

# ========== RUN ==========
keywords = ["Rings", "Earring", "Necklace"]
all_data = []

for keyword in keywords:
    print(f"\n🔎 Searching for {keyword}...")
    search(keyword)
    all_data.extend(collect_images(keyword, max_scrolls=3, max_images=5))

driver.quit()

# ========== SAVE TO CSV ==========
with open(CSV_PATH, "w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    writer.writerow(["Keyword", "ImageURL", "SavedFilename"])
    writer.writerows(all_data)

print(f"\n✅ URLs and filenames saved to {CSV_PATH}")



🔎 Searching for Rings...
✅ Saved Rings_0.jpg for Rings
✅ Saved Rings_1.jpg for Rings
✅ Saved Rings_2.jpg for Rings
✅ Saved Rings_3.jpg for Rings
✅ Saved Rings_4.jpg for Rings

🔎 Searching for Earring...
✅ Saved Earring_0.jpg for Earring
✅ Saved Earring_1.jpg for Earring
✅ Saved Earring_2.jpg for Earring
✅ Saved Earring_3.jpg for Earring
✅ Saved Earring_4.jpg for Earring

🔎 Searching for Necklace...
✅ Saved Necklace_0.jpg for Necklace
✅ Saved Necklace_1.jpg for Necklace
✅ Saved Necklace_2.jpg for Necklace
✅ Saved Necklace_3.jpg for Necklace
✅ Saved Necklace_4.jpg for Necklace

✅ URLs and filenames saved to E:\Funndynamix\04_Project Fusion\v14Images\bluestone_urls.csv


its downloading images of alone product and image of product+model which is saved as not good image but file is created.
only save image with actual content of alone product and remove image of baner and ads

#### v15 no model_product, only product and broken image
Product‑alone images usually have a URL pattern like /jewellery/... or contain product in the path.\
Lifestyle/model images often include model, look, or banner in the filename or URL.\
You can filter by these patterns before saving.

In [9]:
import os
import csv
import time
import requests
from dotenv import load_dotenv
load_dotenv()
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.common.exceptions import StaleElementReferenceException
from webdriver_manager.chrome import ChromeDriverManager

# ========== CONFIG ==========
BASE_DIR = r"E:\Funndynamix\04_Project Fusion\v15Images"
os.makedirs(BASE_DIR, exist_ok=True)
CSV_PATH = os.path.join(BASE_DIR, "bluestone_urls.csv")

# ========== HELPERS ==========
def is_valid_image(url):
    """Check if URL points to a real image."""
    try:
        r = requests.head(url, timeout=5)
        return r.status_code == 200 and "image" in r.headers.get("Content-Type", "")
    except Exception:
        return False

def save_image(src, index, keyword):
    """Download image into keyword folder and return filename."""
    keyword_dir = os.path.join(BASE_DIR, keyword)
    os.makedirs(keyword_dir, exist_ok=True)

    filename = f"{keyword}_{index}.jpg"
    file_path = os.path.join(keyword_dir, filename)

    response = requests.get(src, timeout=10)
    if response.status_code == 200:
        with open(file_path, "wb") as f:
            f.write(response.content)
        print(f"✅ Saved {filename} for {keyword}")
        return filename
    else:
        print(f"❌ Failed to download {src}")
        return None

# ========== SELENIUM SETUP ==========
chrome_options = Options()
chrome_options.add_argument("--disable-notifications")
driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=chrome_options)
driver.maximize_window()
driver.get("https://www.bluestone.com/search")
driver.implicitly_wait(5)

def search(keyword):
    count=0
    while True:
        count+=1
        search_input = driver.find_element(By.XPATH, "/html/body/header/div/div[1]/div[2]/div/div/div[2]/div/div/div/div/form/input[6]")
        search_input.send_keys(keyword, Keys.ENTER)
        if count==5:
            break

def collect_images(keyword, max_scrolls=3, max_images=5):
    # Scroll to load more products
    last_height = driver.execute_script("return document.body.scrollHeight")
    for _ in range(max_scrolls):
        driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
        time.sleep(2)
        new_height = driver.execute_script("return document.body.scrollHeight")
        if new_height == last_height:
            break
        last_height = new_height

    img_elements = driver.find_elements(By.CSS_SELECTOR, "ul.product-grid li img")
    results = []
    count = 0
    for index, img in enumerate(img_elements):
        try:
            src = (
                img.get_attribute("src")
                or img.get_attribute("data-src")
                or img.get_attribute("data-original")
            )
            if not src:
                continue

            # Skip lifestyle/model/banner images
            if any(word in src.lower() for word in ["model", "look", "banner"]):
                continue

            # Only save valid product-alone images
            if src.endswith((".jpg", ".jpeg", ".png")) and is_valid_image(src):
                filename = save_image(src, index, keyword)
                if filename:
                    results.append([keyword, src, filename])
                count += 1
                if count >= max_images:
                    break
        except StaleElementReferenceException:
            continue
    return results

# ========== RUN ==========
keywords = ["Rings", "Earring", "Necklace"]
all_data = []

for keyword in keywords:
    print(f"\n🔎 Searching for {keyword}...")
    search(keyword)
    all_data.extend(collect_images(keyword, max_scrolls=3, max_images=5))

driver.quit()

# ========== SAVE TO CSV ==========
with open(CSV_PATH, "w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    writer.writerow(["Keyword", "ImageURL", "SavedFilename"])
    writer.writerows(all_data)

print(f"\n✅ URLs and filenames saved to {CSV_PATH}")



🔎 Searching for Rings...
✅ Saved Rings_0.jpg for Rings
✅ Saved Rings_1.jpg for Rings
✅ Saved Rings_2.jpg for Rings
✅ Saved Rings_3.jpg for Rings
✅ Saved Rings_9.jpg for Rings

🔎 Searching for Earring...
✅ Saved Earring_0.jpg for Earring
✅ Saved Earring_1.jpg for Earring
✅ Saved Earring_2.jpg for Earring
✅ Saved Earring_3.jpg for Earring
✅ Saved Earring_9.jpg for Earring

🔎 Searching for Necklace...
✅ Saved Necklace_0.jpg for Necklace
✅ Saved Necklace_1.jpg for Necklace
✅ Saved Necklace_2.jpg for Necklace
✅ Saved Necklace_3.jpg for Necklace
✅ Saved Necklace_9.jpg for Necklace

✅ URLs and filenames saved to E:\Funndynamix\04_Project Fusion\v15Images\bluestone_urls.csv


#### v16 for giva

In [7]:
import os
from dotenv import load_dotenv
load_dotenv()
import csv
import time
import requests
from urllib.parse import urlparse, urlunparse
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.common.exceptions import StaleElementReferenceException
from webdriver_manager.chrome import ChromeDriverManager

# ========== CONFIG ==========
BASE_DIR = r"E:\Funndynamix\04_Project Fusion\v16images"
website="https://www.giva.co/"
os.makedirs(BASE_DIR, exist_ok=True)
CSV_PATH = os.path.join(BASE_DIR, "giva_urls.csv")

# ========== HELPERS ==========
def is_valid_image(url):
    """Check if URL points to a real image."""
    try:
        r = requests.head(url, timeout=5)
        return r.status_code == 200 and "image" in r.headers.get("Content-Type", "")
    except Exception:
        return False

def clean_url(url):
    """Strip query params from image URL."""
    parsed = urlparse(url)
    return urlunparse(parsed._replace(query=""))

def save_image(src, index, keyword):
    """Download image into keyword folder and return filename."""
    keyword_dir = os.path.join(BASE_DIR, keyword)
    os.makedirs(keyword_dir, exist_ok=True)

    filename = f"{keyword}_{index}.jpg"
    file_path = os.path.join(keyword_dir, filename)

    response = requests.get(src, timeout=10)
    if response.status_code == 200:
        with open(file_path, "wb") as f:
            f.write(response.content)
        print(f"✅ Saved {filename} for {keyword}")
        return filename
    else:
        print(f"❌ Failed to download {src}")
        return None

# ========== SELENIUM SETUP ==========
chrome_options = Options()
chrome_options.add_argument("--disable-notifications")
driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=chrome_options)
driver.maximize_window()
driver.get(website)
driver.implicitly_wait(5)

def search(keyword):
    search_box = driver.find_element(By.CSS_SELECTOR, "input[name='q']")
    search_box.clear()
    search_box.send_keys(keyword, Keys.ENTER)
    time.sleep(3)

def collect_images(keyword, max_scrolls=3, max_images=5):
    # Scroll to load more products
    last_height = driver.execute_script("return document.body.scrollHeight")
    for _ in range(max_scrolls):
        driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
        time.sleep(2)
        new_height = driver.execute_script("return document.body.scrollHeight")
        if new_height == last_height:
            break
        last_height = new_height

    # CSS selector for product-only image (first <img> inside product card)
    img_elements = driver.find_elements(By.CSS_SELECTOR,
        "ul li div:nth-child(2) div div:first-child div:first-child img:first-child"
    )

    results = []
    count = 0
    for index, img in enumerate(img_elements):
        try:
            src = img.get_attribute("src")
            if not src:
                continue

            # Clean URL (remove ?v=...&width=...)
            src = clean_url(src)

            if src.endswith((".jpg", ".jpeg", ".png")) and is_valid_image(src):
                filename = save_image(src, index, keyword)
                if filename:
                    results.append([keyword, src, filename])
                count += 1
                if count >= max_images:
                    break
        except StaleElementReferenceException:
            continue
    return results

# ========== RUN ==========
keywords = ["trending Rings 2025", "woman Earrings", "Necklaces"]
all_data = []

for keyword in keywords:
    print(f"\n🔎 Searching for {keyword}...")
    search(keyword)
    all_data.extend(collect_images(keyword, max_scrolls=3, max_images=5))

driver.quit()

# ========== SAVE TO CSV ==========
with open(CSV_PATH, "w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    writer.writerow(["Keyword", "ImageURL", "SavedFilename"])
    writer.writerows(all_data)

print(f"\n✅ URLs and filenames saved to {CSV_PATH}")



🔎 Searching for trending Rings 2025...
✅ Saved trending Rings 2025_0.jpg for trending Rings 2025
✅ Saved trending Rings 2025_1.jpg for trending Rings 2025
✅ Saved trending Rings 2025_2.jpg for trending Rings 2025
✅ Saved trending Rings 2025_3.jpg for trending Rings 2025
✅ Saved trending Rings 2025_4.jpg for trending Rings 2025

🔎 Searching for woman Earrings...
✅ Saved woman Earrings_0.jpg for woman Earrings
✅ Saved woman Earrings_1.jpg for woman Earrings
✅ Saved woman Earrings_2.jpg for woman Earrings
✅ Saved woman Earrings_3.jpg for woman Earrings
✅ Saved woman Earrings_4.jpg for woman Earrings

🔎 Searching for Necklaces...
✅ Saved Necklaces_0.jpg for Necklaces
✅ Saved Necklaces_1.jpg for Necklaces
✅ Saved Necklaces_2.jpg for Necklaces
✅ Saved Necklaces_3.jpg for Necklaces
✅ Saved Necklaces_4.jpg for Necklaces

✅ URLs and filenames saved to E:\Funndynamix\04_Project Fusion\v16images\giva_urls.csv


#### v17 malabar

In [ ]:
import os
from dotenv import load_dotenv
load_dotenv()
import csv
import time
import requests
from urllib.parse import urlparse, urlunparse
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options  import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.common.exceptions import StaleElementReferenceException
from webdriver_manager.chrome import ChromeDriverManager

# ========== CONFIG ==========
BASE_DIR = r"E:\Funndynamix\04_Project Fusion\v17images"
website="https://www.giva.co/" #"https://www.malabargoldanddiamonds.com/"
os.makedirs(BASE_DIR, exist_ok=True)
CSV_PATH = os.path.join(BASE_DIR, "malabar_urls.csv")

# ========== HELPERS ==========
def is_valid_image(url):
    """Check if URL points to a real image."""
    try:
        r = requests.head(url, timeout=5)
        return r.status_code == 200 and "image" in r.headers.get("Content-Type", "")
    except Exception:
        return False

def clean_url(url):
    """Strip query params from image URL."""
    parsed = urlparse(url)
    return urlunparse(parsed._replace(query=""))

def save_image(src, index, keyword):
    """Download image into keyword folder and return filename."""
    keyword_dir = os.path.join(BASE_DIR, keyword)
    os.makedirs(keyword_dir, exist_ok=True)

    filename = f"{keyword}_{index}.jpg"
    file_path = os.path.join(keyword_dir, filename)

    response = requests.get(src, timeout=10)
    if response.status_code == 200:
        with open(file_path, "wb") as f:
            f.write(response.content)
        print(f"✅ Saved {filename} for {keyword}")
        return filename
    else:
        print(f"❌ Failed to download {src}")
        return None

# ========== SELENIUM SETUP ==========
chrome_options = Options()
chrome_options.add_argument("--disable-notifications")
driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=chrome_options)
driver.maximize_window()
driver.get(website)
driver.implicitly_wait(5)

def search(keyword):
    search_box = driver.find_element(By.CSS_SELECTOR, "input[name='q']")
    search_box.clear()
    search_box.send_keys(keyword, Keys.ENTER)
    time.sleep(3)

def collect_images(keyword, max_scrolls=3, max_images=5):
    # Scroll to load more products
    last_height = driver.execute_script("return document.body.scrollHeight")
    for _ in range(max_scrolls):
        driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
        time.sleep(2)
        new_height = driver.execute_script("return document.body.scrollHeight")
        if new_height == last_height:
            break
        last_height = new_height

    # CSS selector for product-only image (first <img> inside product card)
    img_elements = driver.find_elements(By.CSS_SELECTOR,
        "ul li a img:first-of-type"
    )

    results = []
    count = 0
    for index, img in enumerate(img_elements):
        try:
            src = img.get_attribute("src")
            if not src:
                continue

            # Clean URL (remove ?v=...&width=...)
            src = clean_url(src)

            if src.endswith((".jpg", ".jpeg", ".png")) and is_valid_image(src):
                filename = save_image(src, index, keyword)
                if filename:
                    results.append([keyword, src, filename])
                count += 1
                if count >= max_images:
                    break
        except StaleElementReferenceException:
            continue
    return results

# ========== RUN ==========
keywords = ["trending Rings 2025", "woman Earrings", "Necklace"]
all_data = []

for keyword in keywords:
    print(f"\n🔎 Searching for {keyword}...")
    search(keyword)
    all_data.extend(collect_images(keyword, max_scrolls=3, max_images=5))

driver.quit()

# ========== SAVE TO CSV ==========
with open(CSV_PATH, "w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    writer.writerow(["Keyword", "ImageURL", "SavedFilename"])
    writer.writerows(all_data)

print(f"\n✅ URLs and filenames saved to {CSV_PATH}")



🔎 Searching for trending Rings 2025...

🔎 Searching for woman Earrings...

🔎 Searching for Necklace...

✅ URLs and filenames saved to E:\Funndynamix\04_Project Fusion\v17images\malabar_urls.csv


not working, pulling wrong images as their search bar is also not working propoerly for themselves


#### v18 thought for agno+selenium, but now trying out googlesearch 

google uses selenium\
search will be across the website\
and pull image url\
should use llm for detailed prompt and wants


In [ ]:
from google_images_search import GoogleImagesSearch
import json

# Replace with your actual credentials
GCS_DEVELOPER_KEY = ""
GCS_CX = ""

def get_jewelry_images_1(query: str, time_range: str = 'd', count: int = 10) -> str:
    """
    Fetches image URLs using Google Custom Search API.
    time_range options: 'd' (day), 'w' (week), 'm' (month), 'y' (year)
    """
    gis = GoogleImagesSearch(GCS_DEVELOPER_KEY, GCS_CX)

    # Google uses 'dateRestrict' for time filters
    # format: 'qdr:d' (day), 'qdr:w' (week), etc.
    _search_params = {
        'q': query,
        'num': count,
        'searchType': 'image',
        'dateRestrict': f'qdr:{time_range}' if time_range else None
    }

    try:
        gis.search(search_params=_search_params)
        results = [image.url for image in gis.results()]
        return json.dumps(results)
    except Exception as e:
        print(f"Error: {e}")
        return json.dumps([])
# f = get_jewelry_images("Turkish plain gold bangle in UAE")
f

#### v19 ScrapeGraph API +agno

bcz

| Tool / Library      | Agentic AI                   | Scraping/Web                    | Google-centric     |
| ------------------- | ---------------------------- | ------------------------------- | ------------------ |
| **Google ADK**      | ✅ Agent framework            | ⬜ Needs scraper plugin          | ✅ Yes              |
| **ScrapeGraphAI**   | ✅ Agentic scraping           | ✅ Built-in                      | ⬜ Not Google       |
| **AgenticSearch**   | ✅ Agentic search             | Partial (via search & scraping) | Uses Google Search |
| **Agentic Browser** | ✅ Agentic browser automation | ✅ Yes                           | ⬜ Not Google       |
| **LangChain**       | ✅ Agent framework            | Partial (via add-ons)           | Compatible         |

https://chatgpt.com/c/69577b15-5040-8320-9102-28fea8b4dd5c
https://github.com/luminati-io/web-scraping-with-scrapegraphai

In [ ]:
from agno.agent import Agent
from agno.models.openai import OpenAIChat
from dotenv import load_dotenv
load_dotenv()
from agno.tools.scrapegraph import ScrapeGraphTools

#30/50 credits remaining in scrapegraph account -10 credits per scrape
# ================= CONFIG =================
SGAI_APIKEY =""
OPENAI_API_KEY=""
# === MODEL ===
model = OpenAIChat(
    id="gpt-4o-mini",  # fast + cheap, good for extraction
    api_key=OPENAI_API_KEY
)

# === SCRAPEGRAPH TOOL ===
scrapegraph = ScrapeGraphTools(
    enable_scrape=True,
    api_key=SGAI_APIKEY
)

# === AGENT ===
agent = Agent(
    model=model,
    tools=[scrapegraph],
    markdown=True
)

# === PROMPT ===
agent.print_response(
    """
    Use smartscraper to extract product information from https://www.giva.co/.

    Extract:
    - Product Title
    - Price (numeric value + currency)
    - Primary Product Image URL (main visible image, not thumbnail)
    - Category (Ring, Earring, Necklace, etc.)

    Rules:
    - If multiple products exist, extract first 5
    - Prefer high-resolution image URLs
    - Output strictly as a markdown table
    """,
    stream=True
)


💬 2026-01-06 13:12:50,996 🔑 Initializing Client
💬 2026-01-06 13:12:50,997 ✅ Client initialized successfully


Output()

💬 2026-01-06 13:12:54,416 🔍 Starting smartscraper request
💬 2026-01-06 13:12:54,417 🚀 Making POST request to https://api.scrapegraphai.com/v1/smartscraper


💬 2026-01-06 13:13:17,130 ✅ Request completed successfully: POST https://api.scrapegraphai.com/v1/smartscraper
💬 2026-01-06 13:13:17,131 ✨ Smartscraper request completed successfully


#### v20 duckduckgo + agno

In [ ]:
from agno.agent import Agent
from agno.models.google import Gemini
from dotenv import load_dotenv   
import os  
load_dotenv()

# Create an Agno agent with an image-capable Gemini model
GEMINI_API_KEY = ""

# Initialize the agent with Gemini model
agent = Agent(
    model=Gemini(api_key=GEMINI_API_KEY),
    instructions="You are a helpful assistant."
)

# Run a simple query
response = agent.run("What is the capital of delhi?")
print(response.content)

In [ ]:
from agno.agent import Agent
from agno.models.google import Gemini as AgnoGemini
from dotenv import load_dotenv   
import os  
load_dotenv()
GEMINI_API_KEY = ""
agno_agent = Agent(
    model=AgnoGemini(
        id="gemini-2.5-flash",
        api_key=GEMINI_API_KEY,     
    ),
    #Act Senior jewelry market intelligence.
    instructions="""
    You are a senior jewelry market intelligence expert. 
    Given a jewelry keyword: 
    - Identify high-performing market trends 
    - Buyer intent variations 
    - Popular design styles currently selling well 
    Output ONLY a Python list of 4–6 concise image search queries. 
    No explanations. No markdown.
    """
)

def agno_generate_search_intents(keyword):
    """
    AGNO converts a raw keyword into
    market-relevant DDGS search intents.
    """
    try:
        # prompt = f"""Only 3 search queries for "{keyword}" No markdown."""
        prompt = f"""Generate high-intent jewelry search queries for "{keyword}" 
        targeting top popular best-selling designs and catalog-quality images. """

        response = agno_agent.run(prompt)
        text = response.content.strip()

        if text.startswith("[") and text.endswith("]"):
            intents = eval(text)
            return [q.strip() for q in intents if isinstance(q, str)]

    except Exception as e:
        print(f"⚠️ AGNO failed for '{keyword}': {e}")

    return [text]

print(agno_generate_search_intents("gold pendant designs"))

In [ ]:
from google import genai
from google.genai import types
from dotenv import load_dotenv   
import os  
load_dotenv()

# Create an Agno agent with an image-capable Gemini model
GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")

# Initialize client
client = genai.Client(api_key=GEMINI_API_KEY)

# Generate image from prompt
response = client.models.generate_images(
    model="imagen-4.0-generate-001",
    prompt="A surreal cat sitting in a tree, vibrant colors",
    config=types.GenerateImagesConfig(
        number_of_images=1,
        include_rai_reason=True
    )
)

# Save or show the image
img_data = response.generated_images[0].image
img_data.show()
img_data.save("images/surreal_cat_tree.jpg")

#do not delete output, cat image


make ring with agno+google gemini

In [ ]:
from agno.agent import Agent, RunOutput
from agno.models.google import Gemini
from PIL import Image
from io import BytesIO
from dotenv import load_dotenv   
import os  
load_dotenv()

# Create an Agno agent with an image-capable Gemini model
GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")

agent = Agent(
    model=Gemini(
        id="gemini-2.5-flash-image",
        response_modalities=["Text", "Image"],
        api_key=GEMINI_API_KEY,
        max_output_tokens=1200
    )
)

# Prompt for image generation
prompt = "A premium gold ring with diamond, studio lighting, white background"

# Run agent
result: RunOutput = agent.run(prompt)

# Handle generated images
if result.images:
    for i, img in enumerate(result.images):
        image = Image.open(BytesIO(img.content))
        image.show()
        image.save(f"images/agno_image_{i}.png")
else:
    print("No image generated")

#do not delete output, image from txt


fuse 2 image 5jan

In [ ]:
from agno.agent import Agent, RunOutput
from agno.models.google import Gemini
from PIL import Image as PILImage
from agno.media import Image
from io import BytesIO
from dotenv import load_dotenv
import os
load_dotenv()

# GEMINI_API_KEY=''  #company'
GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")

# Load two images
with open("/home/ec2-user/fusion_engine/Categorized data/Earring/PSD138-13.jpg", "rb") as f:
    img1_bytes = f.read()

with open("/home/ec2-user/fusion_engine/Categorized data/Earring/PSD139-11.jpg", "rb") as f:
    img2_bytes = f.read()


# Wrap bytes into Agno ImageArtifact
img1 = Image(
    content=img1_bytes,
    mime_type="image/jpg"
)

img2 = Image(
    content=img2_bytes,
    mime_type="image/jpg"
)


agent = Agent(
    model=Gemini(
        id="gemini-2.5-flash-image",
        response_modalities=["Text", "Image"],
        max_output_tokens=1200,
        api_key=GEMINI_API_KEY
    )
)

prompt = """
Fuse the design elements of image 1 and image 2.
- Use the band style from image 1
- Use the stone shape and setting from image 2
- Premium jewelry product photo
- White studio background
"""

# ✅ PASS IMAGE ARTIFACTS
result: RunOutput = agent.run(
    prompt,
    images=[img1, img2]
)

if result.images:
    for i, img in enumerate(result.images):
        image = PILImage.open(BytesIO(img.content))
        image.show()
        image.save(f"images/fused_image_{i}.png")
else:
    print("No image generated")


#do not delete output, fused image

In [ ]:
#5-6jan
import os
import csv
import time
import requests
from urllib.parse import urlparse, urlunparse
from ddgs import DDGS
import google.genai as genai
from dotenv import load_dotenv

# ================= CONFIG =================
load_dotenv()

BASE_DIR = r"/home/ec2-user/fusion_engine/Shreeya-Agno/images"
os.makedirs(BASE_DIR, exist_ok=True)

CSV_PATH = os.path.join(BASE_DIR, "ddgs_images.csv")

GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")
USE_GEMINI_FILTER = True  # turn OFF if not needed

# ================= GEMINI =================
if USE_GEMINI_FILTER:
    gemini=genai.Client(api_key=GEMINI_API_KEY)

def gemini_is_relevant(keyword, image_url):
    """
    Ask Gemini if image matches jewelry intent.
    Very lightweight check.
    """
    try:
        prompt = f"""
        Keyword: {keyword}
        Image URL: {image_url}

        Question:
        Is this image likely a jewelry product image matching the keyword?
        Answer ONLY yes or no.
        """
        r = gemini.models.generate_content(prompt)
        return "yes" in r.text.lower()
    except Exception:
        return True  # fail-open

# ================= HELPERS =================
def clean_url(url):
    parsed = urlparse(url)
    return urlunparse(parsed._replace(query=""))

def is_valid_image(url):
    try:
        r = requests.head(url, timeout=5, allow_redirects=True)
        return r.status_code == 200 and "image" in r.headers.get("Content-Type", "")
    except:
        return False

def save_image(url, keyword, index):
    folder = os.path.join(BASE_DIR, keyword.replace(" ", "_"))
    os.makedirs(folder, exist_ok=True)

    filename = f"{keyword.replace(' ', '_')}_{index}.jpg"
    path = os.path.join(folder, filename)

    r = requests.get(url, timeout=10)
    if r.status_code == 200:
        with open(path, "wb") as f:
            f.write(r.content)
        print(f"✅ Saved {filename}")
        return filename
    return None

# ================= CORE SCRAPER =================
def scrape_images(keyword, max_images=5):
    results = []

    with DDGS() as ddgs:
        images = ddgs.images(
            keyword,
            max_results=30,
            safesearch="moderate",
            size=None,
            type_image=None,
            layout=None,
            license_image=None,
        )

        count = 0
        for img in images:
            src = img.get("image")
            if not src:
                continue

            src = clean_url(src)

            if not src.endswith((".jpg", ".jpeg", ".png")):
                continue

            if not is_valid_image(src):
                continue

            if USE_GEMINI_FILTER:
                if not gemini_is_relevant(keyword, src):
                    continue

            filename = save_image(src, keyword, count)
            if filename:
                results.append([keyword, src, filename])
                count += 1

            if count >= max_images:
                break

            time.sleep(0.5)

    return results

# ================= RUN =================
keywords = [
    "trending gold rings 2025",
    "diamond earrings women",
    "gold necklace design"
]

all_rows = []

for kw in keywords:
    print(f"\n🔍 Searching: {kw}")
    rows = scrape_images(kw, max_images=5)
    all_rows.extend(rows)

# ================= CSV =================
with open(CSV_PATH, "w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    writer.writerow(["Keyword", "SavedFilename", "ImageURL"])
    writer.writerows(all_rows)

print(f"\n✅ Done. CSV saved at {CSV_PATH}")


site focused 6jan

In [ ]:
import os
import csv
import time
import requests
from urllib.parse import urlparse, urlunparse
from ddgs import DDGS
import google.genai as genai
from dotenv import load_dotenv

BASE_DIR = r"/home/ec2-user/fusion_engine/Shreeya-Agno/images/brand_images"
load_dotenv()

os.makedirs(BASE_DIR, exist_ok=True)

CSV_PATH = os.path.join(BASE_DIR, "brand_images.csv")

GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")
USE_GEMINI_FILTER = True  # set False to skip AI filtering

SITES = {
    "malabar": "malabargoldanddiamonds.com",
    "giva": "giva.co",
    "kalyan": "kalyanjewellers.net",
}

KEYWORDS = [
    "gold ring",
    "diamond earrings",
    "gold necklace"
]

MAX_IMAGES_PER_QUERY = 5

# ================= GEMINI =================
if USE_GEMINI_FILTER:
    gemini=genai.Client(api_key=GEMINI_API_KEY)

def gemini_is_jewelry(keyword, image_url):
    try:
        prompt = f"""{keyword}
        Image URL: {image_url}

        Task:
        Decide if the image is a CLEAN, USABLE JEWELRY PRODUCT IMAGE.

        Say "NO" if the image contains ANY of these:
        - banners, posters, ads
        - icons, logos, UI elements
        - text overlays, prices, offers
        - watermarks or brand names
        - collages or multiple products
        - lifestyle shots with models
        - screenshots or website headers
        - low-quality or blurry thumbnails

        Say "YES" ONLY if ALL are true:
        - single jewelry item
        - clear and centered
        - plain or studio background
        - no text or graphics
        - suitable for ecommerce catalog

        Answer ONLY one word:
        YES or NO
        """
        r = gemini.models.generate_content(prompt)
        return "yes" in r.text.lower()
    except:
        return True

# ================= HELPERS =================
def clean_url(url):
    parsed = urlparse(url)
    return urlunparse(parsed._replace(query=""))

def is_valid_image(url):
    try:
        r = requests.head(url, timeout=5, allow_redirects=True)
        return r.status_code == 200 and "image" in r.headers.get("Content-Type", "")
    except:
        return False

def save_image(url, brand, keyword, index):
    brand_dir = os.path.join(BASE_DIR, brand)
    os.makedirs(brand_dir, exist_ok=True)

    filename = f"{brand}_{keyword.replace(' ', '_')}_{index}.jpg"
    path = os.path.join(brand_dir, filename)

    r = requests.get(url, timeout=10)
    if r.status_code == 200:
        with open(path, "wb") as f:
            f.write(r.content)
        print(f"✅ {brand} → {filename}")
        return filename
    return None

# ================= CORE =================
def scrape_brand_images(brand, domain, keyword):
    query = f"site:{domain} {keyword}"
    rows = []
    count = 0

    with DDGS() as ddgs:
        for img in ddgs.images(query, max_results=30):
            src = img.get("image")
            if not src:
                continue

            src = clean_url(src)

            if not src.endswith((".jpg", ".jpeg", ".png")):
                continue

            if not is_valid_image(src):
                continue

            if USE_GEMINI_FILTER:
                if not gemini_is_jewelry(keyword, src):
                    continue

            filename = save_image(src, brand, keyword, count)
            if filename:
                rows.append([brand, keyword, filename, src])
                count += 1

            if count >= MAX_IMAGES_PER_QUERY:
                break

            time.sleep(0.4)

    return rows

# ================= RUN =================
all_rows = []

for brand, domain in SITES.items():
    print(f"\n🏷️ Brand: {brand}")
    for kw in KEYWORDS:
        print(f"   🔍 {kw}")
        all_rows.extend(scrape_brand_images(brand, domain, kw))

# ================= CSV =================
with open(CSV_PATH, "w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    writer.writerow(["Brand", "Keyword", "SavedFilename", "ImageURL"])
    writer.writerows(all_rows)

print(f"\n✅ Finished. CSV saved at {CSV_PATH}")


#### v21 continue of v20,  improvise

correct the code to not be only site dependent but wide web search, \
add agno for market intelligence\
prompts "turkish golden bangles" should be compatible as well\
gemini prompts refinement\
add to remove banner, icons etc 7jan

In [ ]:
#!/usr/bin/env python3
import os
import csv
import time
import requests
from urllib.parse import urlparse, urlunparse
from dotenv import load_dotenv
from ddgs import DDGS
import google.genai as genai
from ddgs import DDGS
from ddgs.exceptions import DDGSException

# ================== CONFIG ==================
load_dotenv()

BASE_DIR = "/home/ec2-user/fusion_engine/Shreeya-Agno/images/v21brand_images"
CSV_PATH = os.path.join(BASE_DIR, "brand_images.csv")
MARKET_INTEL_PATH = os.path.join(BASE_DIR, "market_intelligence.txt")

os.makedirs(BASE_DIR, exist_ok=True)

GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")
USE_GEMINI_FILTER = True
USE_AGNO_MARKET_INTEL = True

MAX_IMAGES_PER_QUERY = 5
MIN_IMAGE_BYTES = 20_000  # removes icons / thumbnails

SITES = {
    "malabar": "malabargoldanddiamonds.com",
    "giva": "giva.co",
    "kalyan": "kalyanjewellers.net",
}

KEYWORDS = [
    "gold ring",
    "turkish golden bangles with diamond studded, filigree design, temple motifs",
]

# ================== GEMINI ==================
if USE_GEMINI_FILTER:
    gemini = genai.Client(api_key=GEMINI_API_KEY)

def gemini_is_clean_jewelry(keyword, image_url):
    try:
        prompt = f"""
        Keyword / Prompt:
        {keyword}

        Image URL:
        {image_url}

        Decide if this image is a CLEAN, SINGLE JEWELRY PRODUCT IMAGE.

        Reject if:
        - banners, posters, ads
        - logos, icons, UI elements
        - text overlays or prices
        - watermarks or brand marks
        - collages or grids
        - lifestyle shots with models
        - screenshots or headers
        - packaging images

        Accept only if:
        - single jewelry item
        - clear focus
        - studio/plain background
        - catalog-quality

        Answer ONLY:
        YES or NO
        """
        r = gemini.models.generate_content(prompt)
        return "yes" in r.text.lower()
    except Exception:
        return True  # fail-open
    
def clean_url(url):
    parsed = urlparse(url)
    return urlunparse(parsed._replace(query=""))

def looks_like_ui_asset(url):
    bad_tokens = [
        "logo", "icon", "sprite", "banner", "header",
        "footer", "menu", "placeholder", "thumb"
    ]
    u = url.lower()
    return any(tok in u for tok in bad_tokens)

def is_valid_image(url):
    try:
        r = requests.head(url, timeout=5, allow_redirects=True)
        if r.status_code != 200:
            return False
        if "image" not in r.headers.get("Content-Type", ""):
            return False
        size = int(r.headers.get("Content-Length", 0))
        return size > MIN_IMAGE_BYTES
    except Exception:
        return False
    
def save_image(url, brand, keyword, index):
    brand = brand or "open_web"
    brand_dir = os.path.join(BASE_DIR, brand)
    os.makedirs(brand_dir, exist_ok=True)

    safe_kw = keyword[:50].replace(" ", "_").replace(",", "")
    filename = f"{brand}_{safe_kw}_{index}.jpg"
    path = os.path.join(brand_dir, filename)

    r = requests.get(url, timeout=10)
    if r.status_code == 200:
        with open(path, "wb") as f:
            f.write(r.content)
        print(f"✅ {brand} → {filename}")
        return filename
    return None
# ================== CORE SCRAPER ==================
def scrape_images(brand, domain, keyword):
    rows = []
    count = 0

    # Primary query (site-specific if available)
    primary_query = f"site:{domain} {keyword}" if domain else keyword
    fallback_query = keyword

    def run_ddgs_query(query):
        nonlocal count
        with DDGS() as ddgs:
            for img in ddgs.images(query, max_results=40):
                src = img.get("image")
                if not src:
                    continue

                src = clean_url(src)

                if looks_like_ui_asset(src):
                    continue

                if not src.lower().endswith((".jpg", ".jpeg", ".png")):
                    continue

                if not is_valid_image(src):
                    continue

                if USE_GEMINI_FILTER:
                    if not gemini_is_clean_jewelry(keyword, src):
                        continue

                filename = save_image(src, brand, keyword, count)
                if filename:
                    rows.append([brand or "open_web", keyword, filename, src])
                    count += 1

                if count >= MAX_IMAGES_PER_QUERY:
                    break

                time.sleep(0.4)

    # ----------------- TRY PRIMARY -----------------
    try:
        run_ddgs_query(primary_query)
    except DDGSException:
        print(f"⚠️ No results for site query → fallback to open web")

    # ----------------- FALLBACK -----------------
    if count < MAX_IMAGES_PER_QUERY:
        try:
            run_ddgs_query(fallback_query)
        except DDGSException:
            print(f"❌ No results even in open web")

    return rows


def agno_market_intelligence_prompt(keyword):
    return f"""
        You are a senior jewelry market intelligence analyst.

        Analyze this jewelry design:
        "{keyword}"

        Answer the following:

        1. Why does this design exist?
        (cultural influence, aesthetics, customer psychology etc)

        2. What design pattern or tradition does it represent?
        (e.g. filigree, temple, Ottoman, bridal, contemporary etc)

        3. How frequently does this design appear in the market?
        (rare / emerging / common / saturated etc)

        4. Where does it sell best?
        - price range
        - ideal occasion (wedding, daily wear, festival, luxury)

        Be precise, structured, and business-focused, one liner, short, point wise.
        """

all_rows = []
market_intel_blocks = []

# Brand-based scraping
for brand, domain in SITES.items():
    print(f"\n🏷️ Brand: {brand}")
    for kw in KEYWORDS:
        print(f"   🔍 {kw}")
        all_rows.extend(scrape_images(brand, domain, kw))

# Open web scraping (no site)
print("\n🌍 Open Web Search")
for kw in KEYWORDS:
    all_rows.extend(scrape_images(None, None, kw))

# Save CSV
with open(CSV_PATH, "w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    writer.writerow(["Brand", "Keyword", "SavedFilename", "ImageURL"])
    writer.writerows(all_rows)

# Generate market intelligence prompts
if USE_AGNO_MARKET_INTEL:
    for kw in KEYWORDS:
        market_intel_blocks.append(agno_market_intelligence_prompt(kw))

    with open(MARKET_INTEL_PATH, "w", encoding="utf-8") as f:
        f.write("\n\n---\n\n".join(market_intel_blocks))

print("\n✅ DONE")
print(f"📄 CSV → {CSV_PATH}")
print(f"🧠 Market Intel Prompts → {MARKET_INTEL_PATH}")


#### v22 =v21+agno
v21 + agno market intelligence + ddgs  + whole trend pulls out /newly updated and then agno to give text that why its relevant to scrape

7-8jan

In [ ]:
#!/usr/bin/env python3

import os
import csv
import time
import requests
from urllib.parse import urlparse, urlunparse
from dotenv import load_dotenv
load_dotenv()
from ddgs import DDGS
from ddgs.exceptions import DDGSException
import google.genai as genai

from agno.agent import Agent
from agno.models.google import Gemini as AgnoGemini

# ================== INPUTS ==================

SITES = {
    # "malabar": "malabargoldanddiamonds.com",
    "giva": "giva.co",
    # "kalyan": "kalyanjewellers.net",
}

KEYWORDS = [
    "gold ring",
]

#Agno Market Intelligence, check market_intelligence.txt for answers, keep short, 
agnoMI="""purpose,tradition,demand,market,buyer intent,price range in INR,social_media pscyhe
    Provide short, one liners for each, business focused"""
#Total scraped images = num_of_generated_prompts X MAX_IMAGES_PER_QUERY
num_of_generated_prompts = 2
MAX_IMAGES_PER_QUERY = 5

# ================== CONFIG ==================

BASE_DIR = "/home/ec2-user/fusion_engine/Shreeya-Agno/images/v22brand_images"
CSV_PATH = os.path.join(BASE_DIR, "brand_images.csv")
MARKET_INTEL_PATH = os.path.join(BASE_DIR, "market_intelligence.txt")

os.makedirs(BASE_DIR, exist_ok=True)

GEMINI_API_KEY = ""

USE_GEMINI_FILTER = True
USE_AGNO_MARKET_INTEL = True

MIN_IMAGE_BYTES = 20_000  # removes icons / thumbnails

# ================== GEMINI (Image Gate) ==================
if USE_GEMINI_FILTER:
    gemini = genai.Client(api_key=GEMINI_API_KEY)

def gemini_is_clean_jewelry(keyword, image_url):
    try:
        prompt = f"""
        Keyword: {keyword}
        Image URL:{image_url}

        Is this a single jewelry product? Reject ads,text,logos,models,collages,banner. Accept single jewelry item. YES/NO
        """
        r = gemini.models.generate_content(prompt)
        return "yes" in r.text.lower()
    except Exception:
        return True  # fail-open

# ================== AGNO (Market Intelligence) ==================
agno_agent = Agent(
    model=AgnoGemini(
        id="gemini-2.5-flash",
        api_key=GEMINI_API_KEY,
    ),
    instructions="""
    Act senior jewelry intelligence design. 
    Ranging under 4 words
    """
)

def agno_generate_search_intents(keyword,num):
    """
    AGNO converts a raw keyword into
    market-relevant DDGS search intents.
    """
    try:
        #due to constraints of only processing shorter prompts
        prompt = f"""Only {num} short (ranging under 4 word each) search queries for "{keyword}" No markdown. in python list"""
        response = agno_agent.run(prompt)
        text = response.content.strip()

        if text.startswith("[") and text.endswith("]"):
            intents = eval(text)
            return [q.strip() for q in intents if isinstance(q, str)]

    except Exception as e:
        print(f"⚠️ AGNO failed for '{keyword}': {e}")

    return [text]  # fallback

# ================== HELPERS ==================
def clean_url(url):
    parsed = urlparse(url)
    return urlunparse(parsed._replace(query=""))

def looks_like_ui_asset(url):
    bad_tokens = [
        "logo", "icon", "sprite", "banner", "header",
        "footer", "menu", "placeholder", "thumb"
    ]
    return any(tok in url.lower() for tok in bad_tokens)

def is_valid_image(url):
    try:
        r = requests.head(url, timeout=5, allow_redirects=True)
        if r.status_code != 200:
            return False
        if "image" not in r.headers.get("Content-Type", ""):
            return False
        size = int(r.headers.get("Content-Length", 0))
        return size > MIN_IMAGE_BYTES
    except Exception:
        return False

def save_image(url, brand, keyword, index):
    brand = brand or "open_web"
    brand_dir = os.path.join(BASE_DIR, brand)
    os.makedirs(brand_dir, exist_ok=True)

    safe_kw = keyword[:50].replace(" ", "_").replace(",", "")
    filename = f"{brand}_{safe_kw}_{index}.jpg"
    path = os.path.join(brand_dir, filename)

    r = requests.get(url, timeout=10)
    if r.status_code == 200:
        with open(path, "wb") as f:
            f.write(r.content)
        print(f"✅ {brand} → {filename}")
        return filename
    return None

# ================== CORE SCRAPER ================== scraper jpg,png,jpeg
def scrape_images(brand, domain, keyword):
    rows = []
    count = 0

    primary_query = f"site:{domain} {keyword}" if domain else keyword
    fallback_query = keyword

    def run_ddgs_query(query):
        nonlocal count
        with DDGS() as ddgs:
            for img in ddgs.images(query, max_results=40):
                src = img.get("image")
                if not src:
                    continue

                src = clean_url(src)

                if looks_like_ui_asset(src):
                    continue

                if not src.lower().endswith((".jpg", ".jpeg", ".png")):
                    continue

                if not is_valid_image(src):
                    continue

                if USE_GEMINI_FILTER:
                    if not gemini_is_clean_jewelry(keyword, src):
                        continue

                filename = save_image(src, brand, keyword, count)
                if filename:
                    rows.append([brand or "open_web", keyword, filename, src])
                    count += 1

                if count >= MAX_IMAGES_PER_QUERY:
                    break

                time.sleep(0.4)

    try:
        run_ddgs_query(primary_query)
    except DDGSException:
        print("⚠️ Site query failed, falling back")

    if count < MAX_IMAGES_PER_QUERY:
        try:
            run_ddgs_query(fallback_query)
        except DDGSException:
            print("❌ Open web failed")

    return rows

# ================== MARKET INTEL TEXT ==================
def agno_market_intelligence_prompt(keyword,agnoMI):
    prompt = f"""
    Analyze {keyword}: {agnoMI}
    """
    try:
        response = agno_agent.run(prompt)
        # AGNO response object contains .content
        return response.content.strip()
    except Exception as e:
        print(f"⚠️ AGNO failed for '{keyword}': {e}")
        # fallback to template
        return f"""
    🔹 Product: {keyword}
    • Purpose: Unknown
    • Tradition: Unknown
    • Market demand: Unknown
    • Demand: Unknown
    • Buyer intent: Unknown
    """

# ================== RUN ==================
all_rows = []
market_intel_blocks = []

# Brand-based scraping
if SITES:
    for brand, domain in SITES.items():
        print(f"\n🏷️ Brand: {brand}")
        for kw in KEYWORDS:
            print(f"   🧠 AGNO → {kw}")
            intents = agno_generate_search_intents(kw,num_of_generated_prompts)

            for intent in intents:
                print(f"      🔍 {intent}")
                all_rows.extend(scrape_images(brand, domain, intent))
else:
    print("\n⚠️ No brands configured, skipping to open web only.")
    # Open web scraping
    print("\n🌍 Open Web")
    for kw in KEYWORDS:
        intents = agno_generate_search_intents(kw,num_of_generated_prompts)
        for intent in intents:
            all_rows.extend(scrape_images(None, None, intent))

# Save CSV
with open(CSV_PATH, "w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    writer.writerow(["Brand", "Keyword", "SavedFilename", "ImageURL"])
    writer.writerows(all_rows)

# Save market intelligence text
if USE_AGNO_MARKET_INTEL:
    for kw in intents:
        market_intel_blocks.append(agno_market_intelligence_prompt(kw, agnoMI))

    with open(MARKET_INTEL_PATH, "w", encoding="utf-8") as f:
        f.write("\n\n---\n\n".join(market_intel_blocks))

print("\n✅ DONE")
print(f"📄 CSV → {CSV_PATH}")
print(f"🧠 Market Intel → {MARKET_INTEL_PATH}")


ModuleNotFoundError: No module named 'google.genai'